# Image Completion using Context Encoders
### Interactive Demonstration & Evaluation Notebook

**Author:** Ravikant  
**Institution:** VIT Bhopal University  
**Course:** Computer Vision  

---
## 1. Introduction
Image completion (or inpainting) fills in missing/damaged regions of an image. In this notebook, we demonstrate the **Context Encoder** architecture: an Encoder-Decoder deep neural network paired with a PatchGAN discriminator that learns both pixel-wise reconstruction and semantic feature synthesis.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import cv2

# Add src directory to module path
sys.path.append(".")
from src.model import build_context_encoder
from src.dataset import apply_center_mask, apply_random_mask
from src.utils import compute_psnr, compute_ssim, denormalize

print("TensorFlow Version:", tf.__version__)
print("GPUs Available:", len(tf.config.list_physical_devices("GPU")))

## 2. Model Architecture
We initialize the Context Encoder (Generator) with 64x64x3 resolution.

In [ ]:
model = build_context_encoder(image_size=64, channels=3, base_filters=64)
model.summary()

## 3. Sample Generation & Corruption Masking
We create sample images and apply central box masking.

In [ ]:
# Create test samples (geometric patterns and gradients)
samples = []
x = np.linspace(-1, 1, 64)
y = np.linspace(-1, 1, 64)
xx, yy = np.meshgrid(x, y)

# Sample 1: Gradient
s1 = np.zeros((64, 64, 3), dtype=np.float32)
s1[:, :, 0] = xx; s1[:, :, 1] = yy; s1[:, :, 2] = np.sin(xx * 3.14)
samples.append(s1)

# Sample 2: Concentric shapes
s2 = np.zeros((64, 64, 3), dtype=np.float32)
cv2.circle(s2, (32, 32), 24, (0.8, -0.2, 0.5), -1)
cv2.circle(s2, (32, 32), 12, (-0.6, 0.9, -0.3), -1)
samples.append(s2)

originals = np.array(samples, dtype=np.float32)

# Apply center mask (32x32)
corrupted_list, masks_list = [], []
for img in originals:
    corr, _, mask = apply_center_mask(tf.constant(img), mask_size=32)
    corrupted_list.append(corr.numpy())
    masks_list.append(mask.numpy())

corrupted = np.array(corrupted_list)
masks = np.array(masks_list)

print("Input batch shape:", corrupted.shape)

## 4. Inpainting Inference & Visual Evaluation
Feed corrupted inputs into the Context Encoder and compute metrics.

In [ ]:
# Load weights if available
ckpt = tf.train.Checkpoint(context_encoder=model)
if os.path.isdir("checkpoints") and tf.train.latest_checkpoint("checkpoints"):
    ckpt.restore(tf.train.latest_checkpoint("checkpoints")).expect_partial()
    print("Loaded checkpoint.")

# Inference
reconstructed = model(corrupted, training=False).numpy()
composited = (corrupted * masks) + (reconstructed * (1.0 - masks))

# Metrics
psnr = compute_psnr(originals, composited)
ssim = compute_ssim(originals, composited)
print(f"Evaluation Metrics -> PSNR: {psnr:.2f} dB | SSIM: {ssim:.4f}")

# Multi-panel visualization
fig, axes = plt.subplots(len(originals), 4, figsize=(14, 3.5 * len(originals)))
col_headers = ["Original (Ground Truth)", "Binary Mask", "Corrupted Input", "Inpainted Output"]

for col, header in enumerate(col_headers):
    axes[0, col].set_title(header, fontsize=12, fontweight="bold")

for i in range(len(originals)):
    axes[i, 0].imshow(denormalize(originals[i])); axes[i, 0].axis("off")
    axes[i, 1].imshow(masks[i, :, :, 0], cmap="gray"); axes[i, 1].axis("off")
    axes[i, 2].imshow(denormalize(corrupted[i])); axes[i, 2].axis("off")
    axes[i, 3].imshow(denormalize(composited[i])); axes[i, 3].axis("off")

plt.tight_layout()
plt.show()

## 5. Conclusion
The Context Encoder seamlessly recovers the missing pixel regions. The composite image preserves known context outside the mask while synthesizing visually consistent content inside.